# 1. Domain Feature Engineering

## What is Domain Feature Engineering?

Domain feature engineering is the process of creating meaningful features using knowledge about a particular business area or industry.

Domain knowledge means understanding how a business works, what its data represents, and which factors influence its outcomes.

Instead of creating features randomly, we use business understanding to identify useful relationships between existing columns.

## Why is Domain Knowledge Important?

Consider a telecom company that wants to predict customer churn.

A normal dataset may contain:

- Customer age
- Monthly charges
- Total charges
- Tenure
- Support calls
- Contract type
- Internet service

A data scientist who understands the telecom business may identify additional useful information.

For example:

- Customers with short tenure may be more likely to leave.
- Customers with high monthly charges may become dissatisfied.
- Customers making frequent support calls may experience service problems.
- Customers with long-term contracts may be less likely to leave.

These business observations help us create meaningful features.

## Examples of Domain-Based Features

### 1. Average Monthly Spending

Formula:

Average_Monthly_Spending = TotalCharges / Tenure

Reason:
Helps understand the average amount a customer spends per month.

### 2. Customer Loyalty Group

Logic:

- Tenure below 12 months → New Customer
- Tenure from 12 to 36 months → Regular Customer
- Tenure above 36 months → Loyal Customer

Reason:
Customer relationship duration may influence churn.

### 3. High Value Customer

Logic:

MonthlyCharges greater than a selected business threshold.

Reason:
Identifies customers who generate higher monthly revenue.

### 4. Support Issue Level

Logic:

- 0–2 calls → Low
- 3–5 calls → Medium
- More than 5 calls → High

Reason:
Frequent support requests may indicate customer dissatisfaction.

### 5. Contract Commitment

Logic:

Convert contract types into commitment categories.

Reason:
Customers with longer contracts may have a lower likelihood of leaving.

## Domain Feature Engineering Across Industries

| Industry | Example Feature | Business Purpose |
|---|---|---|
| Banking | Debt-to-Income Ratio | Evaluate financial risk |
| Healthcare | BMI | Understand body measurement relationships |
| E-commerce | Average Order Value | Understand customer spending |
| Telecom | Average Monthly Spending | Understand customer value |
| Retail | Purchase Frequency | Identify customer buying behaviour |
| Insurance | Claim Frequency | Understand claim patterns |

## Important Considerations

1. Understand the business problem before creating features.
2. Use meaningful source columns.
3. Document the feature formula.
4. Check the relationship with the target.
5. Avoid unnecessary or duplicate features.
6. Check whether the feature is available at prediction time.
7. Validate the feature using data and business knowledge.

## Final Rule

A good engineered feature should have a clear business meaning and provide useful information to the machine learning model.

# Demonstration of Domain Feature Engineering

## Business Scenario: Telecom Customer Churn Prediction

A telecom company wants to identify customers who may stop using its services.

The company collects customer information such as:

- Tenure
- Monthly charges
- Total charges
- Support calls
- Contract type
- Internet service
- Churn status

Instead of using only the original columns, we apply telecom business knowledge to create additional features.

## Feature Creation Examples

| New Feature | Source Columns | Feature Logic | Business Reason |
|---|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | TotalCharges / Tenure | Understand average monthly spending |
| Customer_Loyalty_Group | Tenure | Group customers by tenure | Identify customer loyalty |
| High_Value_Customer | MonthlyCharges | MonthlyCharges > 80 | Identify higher monthly revenue customers |
| Support_Issue_Level | SupportCalls | Group support calls | Identify possible service dissatisfaction |
| Contract_Commitment | ContractType | Map contract duration | Understand customer commitment |
| Service_Count | InternetService, PhoneService, OnlineSecurity | Count subscribed services | Understand service adoption |
| Revenue_Category | MonthlyCharges | Categorize monthly revenue | Group customers by spending level |

## Business Interpretation

### Customer A
- Tenure: 5 months
- Monthly Charges: ₹95
- Support Calls: 7
- Contract: Monthly

Possible interpretation:
- New customer.
- Higher monthly spending.
- Frequent support issues.
- Short-term contract.

This customer may require additional attention from the retention team.

### Customer B
- Tenure: 48 months
- Monthly Charges: ₹55
- Support Calls: 1
- Contract: Two Year

Possible interpretation:
- Long-term customer.
- Lower monthly spending.
- Few support issues.
- Long-term contract.

This customer may have a stronger customer relationship.

**Note:** These are business hypotheses, not guaranteed churn outcomes. They must be tested using actual data.

In [1]:
# ============================================================
# NOTEBOOK 14: DOMAIN FEATURE ENGINEERING
# Business Scenario: Telecom Customer Churn Prediction
# ============================================================

# Step 1: Import Libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully!")


# Step 2: Create Telecom Customer Dataset

np.random.seed(42)

n = 200

data = pd.DataFrame({
    "CustomerID": range(1, n + 1),
    "Tenure": np.random.randint(1, 73, n),
    "MonthlyCharges": np.random.uniform(20, 120, n).round(2),
    "SupportCalls": np.random.randint(0, 10, n),
    "ContractType": np.random.choice(
        ["Monthly", "Yearly", "Two Year"], n
    ),
    "InternetService": np.random.choice(
        ["DSL", "Fiber", "No"], n
    ),
    "PhoneService": np.random.choice(
        ["Yes", "No"], n
    ),
    "OnlineSecurity": np.random.choice(
        ["Yes", "No"], n
    )
})

# Calculate Total Charges

data["TotalCharges"] = (
    data["Tenure"] * data["MonthlyCharges"]
).round(2)

# Generate Churn Target

churn_probability = (
    0.15
    + (data["Tenure"] < 12) * 0.20
    + (data["MonthlyCharges"] > 90) * 0.15
    + (data["SupportCalls"] > 5) * 0.20
    + (data["ContractType"] == "Monthly") * 0.15
)

churn_probability = churn_probability.clip(0, 0.95)

data["Churn"] = np.random.binomial(
    1, churn_probability
)

print("\nTelecom Dataset Created Successfully!")
print("\nFirst Five Records:")
print(data.head())

print("\nOriginal Dataset Shape:")
print(data.shape)


# Step 3: Create Average Monthly Spending

data["Average_Monthly_Spending"] = np.where(
    data["Tenure"] > 0,
    data["TotalCharges"] / data["Tenure"],
    0
)


# Step 4: Create Customer Loyalty Group

def loyalty_group(tenure):
    if tenure < 12:
        return "New Customer"
    elif tenure <= 36:
        return "Regular Customer"
    else:
        return "Loyal Customer"


data["Customer_Loyalty_Group"] = (
    data["Tenure"].apply(loyalty_group)
)


# Step 5: Create High Value Customer Feature

data["High_Value_Customer"] = np.where(
    data["MonthlyCharges"] > 80,
    "Yes",
    "No"
)


# Step 6: Create Support Issue Level

def support_issue_level(calls):
    if calls <= 2:
        return "Low"
    elif calls <= 5:
        return "Medium"
    else:
        return "High"


data["Support_Issue_Level"] = (
    data["SupportCalls"].apply(support_issue_level)
)


# Step 7: Create Contract Commitment

contract_mapping = {
    "Monthly": "Short Term",
    "Yearly": "Medium Term",
    "Two Year": "Long Term"
}

data["Contract_Commitment"] = (
    data["ContractType"].map(contract_mapping)
)


# Step 8: Create Service Count

data["Service_Count"] = (
    (data["InternetService"] != "No").astype(int)
    + (data["PhoneService"] == "Yes").astype(int)
    + (data["OnlineSecurity"] == "Yes").astype(int)
)


# Step 9: Create Revenue Category

data["Revenue_Category"] = pd.cut(
    data["MonthlyCharges"],
    bins=[0, 40, 80, 120],
    labels=["Low Revenue", "Medium Revenue", "High Revenue"],
    include_lowest=True
)


# Step 10: Analyze Engineered Features Against Churn

print("\nChurn Rate by Customer Loyalty Group:")

print(
    data.groupby(
        "Customer_Loyalty_Group"
    )["Churn"].mean().round(3)
)

print("\nChurn Rate by Support Issue Level:")

print(
    data.groupby(
        "Support_Issue_Level"
    )["Churn"].mean().round(3)
)

print("\nChurn Rate by Contract Commitment:")

print(
    data.groupby(
        "Contract_Commitment"
    )["Churn"].mean().round(3)
)


# Step 11: Create Final Feature Dataset

engineered_features = [
    "Average_Monthly_Spending",
    "Customer_Loyalty_Group",
    "High_Value_Customer",
    "Support_Issue_Level",
    "Contract_Commitment",
    "Service_Count",
    "Revenue_Category"
]

final_dataset = data[
    ["CustomerID"] + engineered_features + ["Churn"]
].copy()

print("\nFinal Engineered Dataset:")
print(final_dataset.head())

print("\nFinal Dataset Shape:")
print(final_dataset.shape)

print("\nMissing Values:")
print(final_dataset.isnull().sum())


# Step 12: Compare Original and Engineered Features

comparison = pd.DataFrame({
    "Original_Features": [
        "Tenure",
        "MonthlyCharges",
        "SupportCalls",
        "ContractType"
    ],
    "Engineered_Features": [
        "Customer_Loyalty_Group",
        "High_Value_Customer",
        "Support_Issue_Level",
        "Contract_Commitment"
    ]
})

print("\nOriginal vs Engineered Features:")
print(comparison)


# Step 13: Display Final Summary

print("\n========== FINAL SUMMARY ==========")

print("Original Dataset Shape:", data.shape)
print("Final Feature Dataset Shape:", final_dataset.shape)

print("\nNumber of Engineered Features:",
      len(engineered_features))

print("\nEngineered Features:")
for feature in engineered_features:
    print("-", feature)

print("\nDomain Feature Engineering Completed Successfully!")

Libraries imported successfully!

Telecom Dataset Created Successfully!

First Five Records:
   CustomerID  Tenure  MonthlyCharges  SupportCalls ContractType  \
0           1      52           54.11             3      Monthly   
1           2      15           31.35             9      Monthly   
2           3      72          112.47             4       Yearly   
3           4      61          107.73             1      Monthly   
4           5      21           45.79             7       Yearly   

  InternetService PhoneService OnlineSecurity  TotalCharges  Churn  
0             DSL          Yes             No       2813.72      0  
1             DSL          Yes             No        470.25      0  
2           Fiber          Yes            Yes       8097.84      0  
3             DSL           No             No       6571.53      1  
4           Fiber           No            Yes        961.59      0  

Original Dataset Shape:
(200, 10)

Churn Rate by Customer Loyalty Group:
Customer_L

# 4. Feature Justification

## Feature Engineering Documentation

| Feature Name | Source Columns | Feature Logic / Formula | Reason for Creation | Expected Relationship with Churn | Business Meaning | ML Value | Leakage Risk | Final Decision |
|---|---|---|---|---|---|---|---|---|
| Average_Monthly_Spending | TotalCharges, Tenure | TotalCharges / Tenure | Understand average spending | Higher spending may increase churn for some customers | Average monthly revenue per customer | Medium | Low if calculated from data available at prediction time | Retain |
| Customer_Loyalty_Group | Tenure | Group tenure into three categories | Identify customer relationship duration | New customers may have higher churn | Customer loyalty segment | High | Low | Retain |
| High_Value_Customer | MonthlyCharges | MonthlyCharges > 80 | Identify higher revenue customers | Higher charges may increase churn | High monthly revenue segment | Medium | Low | Retain |
| Support_Issue_Level | SupportCalls | Group support calls into Low, Medium, High | Identify customer service problems | More support issues may increase churn | Customer service experience | High | Low | Retain |
| Contract_Commitment | ContractType | Map contract duration into categories | Understand customer commitment | Short-term contracts may have higher churn | Contract relationship | High | Low | Retain |
| Service_Count | InternetService, PhoneService, OnlineSecurity | Count subscribed services | Understand service adoption | Relationship may vary | Number of subscribed services | Medium | Low | Retain |
| Revenue_Category | MonthlyCharges | Divide spending into revenue groups | Segment customers by revenue | Relationship may vary | Customer spending segment | Medium | Low | Retain |

## Business Interpretation

Domain-based features help transform raw customer information into meaningful business indicators.

For example:

- Customer_Loyalty_Group helps identify new and long-term customers.
- Support_Issue_Level helps identify customers who frequently contact customer support.
- High_Value_Customer helps identify customers generating higher monthly revenue.
- Contract_Commitment helps understand the customer's contract duration.

These features can help the telecom company develop targeted customer retention strategies.

## Important Considerations

1. Business assumptions must be validated using real data.
2. Thresholds such as 80 for high monthly charges should be selected using business knowledge or training data.
3. Similar features may contain duplicate information.
4. Features should be tested to understand their contribution to model performance.
5. Features calculated using future information must be removed.
6. Feature definitions should be documented clearly.

## Final Learning

Domain feature engineering connects business understanding with machine learning.

A feature should not be retained simply because it can be created. It should have a meaningful business purpose, be available at prediction time, and provide useful information to the model.

**Final Rule:** Use domain knowledge to create meaningful features, then validate their usefulness using data and machine learning experiments.